# README screenshots

Run each cell in VS Code and screenshot the ones marked 📸, cell and output
together. Save them in `docs/img/` under the name given.

In [ ]:
%load_ext tidy3.jupyter
from tidy3 import *

penguins = scan_csv(
    "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv",
    null_values="NA",
) >> compute()

📸 **`notebook_dark.png`**: this cell and its table.

In [ ]:
# Heaviest groups first
penguins
>> drop_na(sex)                        # 11 penguins have no recorded sex
>> group_by(species, sex)
>> summarise(
    n = n(),
    bill_mm = mean(bill_length_mm).round(1),
    mass_kg = (mean(body_mass_g) / 1000).round(2),
)
>> arrange(desc(mass_kg))

📸 **`reshape.png`**: this cell and its table.

In [ ]:
long = penguins
>> mutate(id = row_number())
>> select(id, species, ends_with("_mm"))
>> pivot_longer(ends_with("_mm"), names_to = "measure", values_to = "mm",
                values_drop_na = True)

long >> head(4)

📸 **`sklearn.png`**: this cell and its table (needs `pip install scikit-learn`).

In [ ]:
from sklearn.cluster import KMeans

features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]
complete = penguins >> drop_na(all_of(features)) >> compute()

X = complete
>> mutate(across(all_of(features), lambda x: (x - mean(x)) / sd(x)))   # R's scale()
>> to_numpy(columns = features)

kmeans = KMeans(n_clusters=3, n_init=10, random_state=1).fit(X)

clustered = complete >> mutate(cluster = kmeans.labels_ + 1)   # labels come back
clustered >> count(species, cluster)